# 04. Offline RL Imitation Cql IQL Gail and Decision Transformer

- **Category:** 08_learning_paradigms_rl_and_alignment / 02_reinforcement_learning_and_world_models
- **Lineage & Citations:** Pomerleau (1988 ALVINN); Ross et al. (2011 DAgger); Ho & Ermon (2016 GAIL); Kumar et al. (2020 CQL); Kostrikov et al. (2021 IQL); Chen et al. (2021 Decision Transformer).
- **Purpose:** First-principles mathematical implementation and empirical verification of offline reinforcement learning and imitation learning algorithms: Conservative Q-Learning (CQL), Implicit Q-Learning (IQL), and Advantage-Weighted Policy Extraction.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [03. Continuous Control Ddpg Td3 and SAC](03_continuous_control_ddpg_td3_and_sac.ipynb) $\to$ `🔗 Downstream Syntheses:` [05. Latent World Models Rssm Dreamer Tdmpc and Rnd Curiosity](05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to Use**  
> Do not apply standard off-policy RL algorithms (such as DQN, SAC, or DDPG) directly to static offline datasets without out-of-distribution correction. Standard Bellman updates greedily query counterfactual actions $\max_{a'} Q(s', a')$, hallucinating massive positive values on out-of-distribution actions. Always use **Conservative Q-Learning (CQL)** or in-sample **Implicit Q-Learning (IQL)** to ensure that value functions remain bounded by the data support.

## Mathematical Foundations & Governing Equations

### 1. Conservative Q-Learning (CQL) Gap Penalty
$$
\mathcal{R}_{\text{CQL}}(Q) = \mathbb{E}_{s \sim \mathcal{D}} \left[ \log \sum_{a} \exp(Q(s, a)) - Q(s, a_{\text{data}}) \right]
$$

### 2. IQL Asymmetric Expectile Regression
$$
L_2^\tau(u) = |\tau - \mathbb{I}(u < 0)| \cdot u^2, \quad u = Q_\theta(s, a) - V_\psi(s)
$$
$$
L_V(\psi) = \mathbb{E}_{(s, a) \sim \mathcal{D}} \left[ L_2^\tau \left( Q_\theta(s, a) - V_\psi(s) \right) \right]
$$

### 3. Advantage-Weighted Policy Extraction (AWR)
$$
\mathcal{L}_\pi(\phi) = \mathbb{E}_{(s, a) \sim \mathcal{D}} \left[ \min\left( \exp\left( \beta (Q(s, a) - V(s)) \right), \; w_{\max} \right) \cdot \| \pi_\phi(s) - a \|_2^2 \right]
$$

### Asymptotic Time & Memory Complexity
- CQL Penalty: $\mathcal{O}(B \cdot |\mathcal{A}|)$ time and memory.
- IQL Expectile Loss: $\mathcal{O}(B)$ elementwise time and memory.
- AWR Actor Extraction: $\mathcal{O}(B \cdot D)$ time and memory.

In [ ]:
# Setup and seed verification environment
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)

# Locate the golden checkpoint
CKPT_REL = Path("weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return x @ w + b

def cql_penalty(q_values: torch.Tensor, dataset_actions: torch.Tensor) -> torch.Tensor:
    logsumexp_q = torch.logsumexp(q_values, dim=-1)
    data_q = torch.gather(q_values, dim=-1, index=dataset_actions.unsqueeze(-1)).squeeze(-1)
    return torch.mean(logsumexp_q - data_q)

def iql_expectile_loss(diff: torch.Tensor, tau: float = 0.7) -> torch.Tensor:
    weight = torch.where(diff > 0, tau, 1.0 - tau)
    return torch.mean(weight * (diff ** 2))

def iql_actor_loss(
    pred_action: torch.Tensor,
    target_action: torch.Tensor,
    adv: torch.Tensor,
    beta: float = 3.0,
    max_weight: float = 100.0,
) -> torch.Tensor:
    weights = torch.clamp(torch.exp(beta * adv), max=max_weight)
    sq_err = torch.sum((pred_action - target_action) ** 2, dim=-1)
    return torch.mean(weights * sq_err)

In [ ]:
# Multi-track numerical parity assertions
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y - weights["test_output"])))
assert err_anchor < 1e-6, f"Linear anchor parity error: {err_anchor:.2e}"

cql_pen = cql_penalty(weights["q_values"], weights["dataset_actions"])
err_cql = float(torch.max(torch.abs(cql_pen - weights["cql_penalty_ref"])))
assert err_cql < 1e-6, f"CQL penalty parity error: {err_cql:.2e}"

iql_exp = iql_expectile_loss(weights["diff"], tau=0.7)
err_iql = float(torch.max(torch.abs(iql_exp - weights["iql_expectile_ref"])))
assert err_iql < 1e-6, f"IQL expectile loss parity error: {err_iql:.2e}"

iql_act = iql_actor_loss(weights["pred_action"], weights["target_action"], weights["adv"], beta=3.0, max_weight=100.0)
err_act = float(torch.max(torch.abs(iql_act - weights["iql_actor_ref"])))
assert err_act < 1e-6, f"IQL actor loss parity error: {err_act:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_cql, err_iql, err_act):.2e})")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_cql = benchmark_fn(
    lambda: cql_penalty(weights["q_values"], weights["dataset_actions"]),
    iters=100,
    name="CQL Gap Penalty",
)
b_iql = benchmark_fn(
    lambda: iql_expectile_loss(weights["diff"]),
    iters=100,
    name="IQL Expectile Loss",
)
b_act = benchmark_fn(
    lambda: iql_actor_loss(weights["pred_action"], weights["target_action"], weights["adv"]),
    iters=100,
    name="IQL Actor Extraction",
)

print(format_benchmark_table([b_cql, b_iql, b_act]))

In [ ]:
# Failure Mode Contrast: OOD Action Exploitation in Naive Offline Q-learning vs CQL
# Suppose 5 actions exist: action 0 was seen in dataset with true return 1.0,
# while actions 1..4 are unseen with true return 0.0, but random function approximation error
# produces an artificial spike of 5.0 on unseen action 3.
q_sample = torch.tensor([[1.0, 0.2, -0.4, 5.0, 0.1]])
dataset_action = torch.tensor([0])

# Naive greedy policy picks max_a Q(s, a):
naive_choice = torch.argmax(q_sample, dim=-1).item()

# CQL penalized Q values:
# CQL penalizes logsumexp(Q) - Q(dataset_action)
penalty = cql_penalty(q_sample, dataset_action).item()

print(f"Naive Selected Action Index: {naive_choice} (Picks unseen OOD action 3 with value 5.0!)")
print(f"CQL Conservative Penalty:    {penalty:.4f} (Heavily penalizes the OOD gap!)")

assert naive_choice == 3, "Naive argmax must pick the erroneous OOD action"
assert penalty > 4.0, f"CQL penalty must reflect the large gap between logsumexp and data action (got {penalty:.2f})"
print("Pathological failure mode contrast successfully verified")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Conservative Value Lower Bounds**: CQL proves that penalizing the gap $\log \sum_a \exp(Q) - Q_{\text{data}}$ creates point-wise lower bounds on expected return, preventing distributional drift.
2. **In-Sample Expectile Magic**: IQL avoids evaluating counterfactual actions altogether by fitting $V(s)$ to upper expectiles of the data using asymmetric $L_2^\tau$ regression.
3. **Decoupled Actor Extraction**: Value estimation and policy extraction are completely decoupled in IQL, eliminating unstable policy-evaluation feedback loops.

---
### Cross-Domain Generalisation Matrix
- **Clinical Healthcare**: IQL derives reliable sepsis management protocols from ICU telemetry logs without querying potentially fatal drug combinations.
- **Autonomous Vehicles**: CQL safely extracts collision avoidance reflexes from human driving logs containing diverse human driver mistakes.
- **Automated E-Commerce**: Advantage-weighted regression ranks product recommendations to maximize long-term customer lifetime value from historical browse logs.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.md](../../../markdown/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.md)
- **PyTorch 2.x Module:** [04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.py](../../../pytorch2/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.py)
- **Functional JAX Module:** [04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.py](../../../jax/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.py)
- **Native Rust Candle Track:** [04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.rs](../../../candle/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.rs)
- **Hardware Kernel (CPU / WGSL):** [04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.rs](../../../kernels/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.rs)
- **Golden SafeTensors:** [04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.safetensors](../../../weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. Continuous Control Ddpg Td3 and SAC](03_continuous_control_ddpg_td3_and_sac.ipynb) | [Master Index](../../../README.md#08_learning_paradigms_rl_and_alignment) | [Next: 05. Latent World Models Rssm Dreamer Tdmpc and Rnd Curiosity](05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.ipynb) &rarr;